# LSTM - Neural Network

## Importing the required libraries and functions. Defining global variables.

In [6]:
import random
import re
from collections import Counter
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import load_dataset
from pandas import DataFrame
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
)
from torch import device, nn
from torch.utils.data import DataLoader, Dataset

LABELS = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}
TOKEN_RE = re.compile(r"[A-Za-z0-9']+")
PAD = "<pad>"
UNK = "<unk>"
MAX_EPOCHS = 20
PATIENCE = 3
LEARNING_RATES = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
CLIP = 1.0

In [7]:
def set_seed(seed: int = 42) -> None:
    """
    Set random seeds for reproducibility. Defaults to 42.

    Args:
        seed (int): Seed value.

    Returns:
        None: Sets the seed to a given value or 42.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

In [8]:
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    print("Using CPU")
    device = torch.device("cpu")

Using CPU


## Loading and preprocessing the dataset

In [9]:
raw = load_dataset("sh0416/ag_news")

split = raw["train"].train_test_split(test_size=0.1, seed=42)
train_set = split["train"]
val_set = split["test"]

test_set = raw["test"]

### Customs functions for text data preprocessing

In [10]:
def tokenize(text: str) -> list:
    """
    Tokenizes the input text into a list of lowercase tokens using a regular expression.

    Args:
        text (str): The input text to tokenize.

    Returns:
        list: A list of lowercase tokens extracted from the input text.
    """
    return TOKEN_RE.findall(text.lower())


def build_vocab(texts, min_freq: int = 2, max_size: int = 30000) -> dict:
    """
    Builds a vocabulary mapping from tokens to integer indices.
    The vocabulary includes only tokens that appear at least `min_freq` times,
    and will be limited to `max_size` tokens (including PAD and UNK).

    Args:
        texts (list): A list of text strings to build the vocabulary from.
        min_freq (int): Minimum frequency for a token to be included in the vocabulary.
        max_size (int): Maximum size of the vocabulary (including PAD and UNK).

    Returns:
        dict: A dictionary mapping tokens to their corresponding integer indices.
    """
    counter = Counter()
    for t in texts:
        counter.update(tokenize(t))
    vocab = {PAD: 0, UNK: 1}

    for word, freq in counter.most_common():
        if freq < min_freq:
            break
        if len(vocab) >= max_size:
            break
        vocab[word] = len(vocab)

    return vocab


def numericalize(tokens: list, vocab: dict) -> list:
    """
    Convert a list of tokens into a list of integer indices using the provided vocabulary.
    Tokens not found in the vocabulary will be mapped to the index of UNK.

    Args:
        tokens (list): A list of tokens to be converted.
        vocab (dict): A dictionary mapping tokens to their corresponding integer indices.

    Returns:
        list: A list of integer indices corresponding to the input tokens.
    """
    return [vocab.get(tok, vocab[UNK]) for tok in tokens]

In [11]:
text = list(train_set["title"]) + list(train_set["description"])
vocab = build_vocab(text, min_freq=2, max_size=30000)
vocab_size = len(vocab)
PAD_IDX = vocab[PAD]
print(vocab_size)

30000


## Defining dataset classes

In [12]:
@dataclass
class Batch:
    """
    A data class representing a batch of data for training or evaluation.
    Attributes:
        x (torch.Tensor): A tensor of shape (B, T) containing token ids.
        lengths (torch.Tensor): A tensor of shape (B,) containing the true lengths of each sequence.
        y (torch.Tensor): A tensor of shape (B,) containing the labels.
    """

    x: torch.Tensor
    lengths: torch.Tensor
    y: torch.Tensor

In [13]:
class TextDataset(Dataset):
    """
    A custom dataset class for handling the specific text data for the task.
    """

    def __init__(self, dataset: dict, vocab: dict, max_len: int = 200) -> None:
        """
        Constructor for the TextDataset class.

        Args:
            dataset (dict): A dataset containing the text data.
            vocab (dict): A dictionary mapping tokens to their corresponding integer indices.
            max_len (int): The maximum length of token sequences. Longer sequences will be truncated.

        Returns:
            None: Initializes the dataset with the provided parameters.
        """
        self.ds = dataset
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self) -> int:
        """
        Returns the number of samples in the dataset.

        Returns:
            int: The number of samples in the dataset.
        """
        return len(self.ds)

    def __getitem__(self, idx: int) -> tuple:
        """
        Returns the token id and label for a given index.

        Args:
            idx (int): The index of the sample to retrieve.

        Returns:
            tuple: A tuple containing the token id and the label.
        """
        item = self.ds[idx]
        tokens = tokenize(item["title"] + " " + item["description"])

        if len(tokens) == 0:
            ids = [self.vocab[UNK]]

        else:
            ids = numericalize(tokens, self.vocab)[: self.max_len]
            if len(ids) == 0:
                ids = [self.vocab[UNK]]

        label = int(item["label"]) - 1
        return ids, label

In [14]:
def collate(batch: list) -> Batch:
    """
    Collate function to convert a list of samples into a batch.

    Args:
        batch (list): A list of samples, where each sample is a tuple of (ids_list, label).

    Returns:
        Batch: A batch containing the padded token ids and corresponding labels.
    """
    lengths = torch.tensor([len(x) for x, _ in batch], dtype=torch.long)
    max_len = int(lengths.max().item()) if len(batch) > 0 else 0
    x = torch.full((len(batch), max_len), vocab[PAD], dtype=torch.long)
    y = torch.tensor([y for _, y in batch], dtype=torch.long)

    for i, (ids, _) in enumerate(batch):
        x[i, : len(ids)] = torch.tensor(ids, dtype=torch.long)

    return Batch(x=x, lengths=lengths, y=y)

## Loading the data

In [15]:
train_loader = DataLoader(
    TextDataset(train_set, vocab), batch_size=64, shuffle=True, collate_fn=collate
)
val_loader = DataLoader(
    TextDataset(val_set, vocab), batch_size=64, shuffle=False, collate_fn=collate
)
test_loader = DataLoader(
    TextDataset(test_set, vocab), batch_size=64, shuffle=False, collate_fn=collate
)

## LSTM model

In [16]:
class LSTMClassifier(nn.Module):
    """
    A class for the LSTM-based text classifier.
    """

    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = 64,
        hidden_dim: int = 64,
        num_layers: int = 2,
        dropout: float = 0.3,
        pad_idx: int = 0,
        num_classes: int = 4,
        bidirectional: bool = False,
    ) -> None:
        """
        Constructor for the LSTMClassifier class.

        Args:
            vocab_size (int): The size of the vocabulary.
            embed_dim (int): The dimension of the word embeddings. Defaults to 64
            hidden_dim (int): The dimension of the hidden state in the LSTM. Defaults to 64
            num_layers (int): The number of layers in the LSTM. Defaults to 2
            dropout (float): The dropout rate. Defaults to 0.3
            pad_idx (int): The index used for padding in the embedding layer. Defaults to 0
            num_classes (int): The number of output classes. Defaults to 4
            bidirectional (bool): Whether to use a bidirectional LSTM. Defaults to False

        Returns:
            None: Initializes the LSTMClassifier with the specified parameters.
        """
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.emb_dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
        )

        rep_dim = hidden_dim * (2 if bidirectional else 1)
        self.rep_dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(rep_dim, num_classes)

    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        """
        Vectorizes the input text and passes it through the LSTM
        and fully connected layers to get the output logits.

        Args:
            x (torch.Tensor): The input tensor containing the tokenized text. Shape: (B, T, E)
            lengths (torch.Tensor): The lengths of the sequences in the batch. Shape: (B,)

        Returns:
            torch.Tensor: The output logits for each class. Shape: (B, num_classes)
        """
        emb = self.emb_dropout(self.embedding(x))
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.lstm(packed)
        h_last = h_n[-1]
        rep = self.rep_dropout(h_last)

        return self.fc(rep)

In [ ]:
def init_model() -> LSTMClassifier:
    """
    Initializes the LSTM model.

    Args:
        None

    Returns:
        LSTMClassifier: The initialized LSTM model.
    """
    return LSTMClassifier(
            vocab_size=vocab_size,
            embed_dim=64,
            hidden_dim=64,
            num_layers=2,
            dropout=0.3,
            pad_idx=PAD_IDX,
        ).to(device)

## Hyperparameter optimization

In [18]:
def tune_lr(learning_rates: list[float], max_epochs: int = 20, patience: int = 3) -> tuple[float, int, DataFrame, dict]:
    """
    Grid search for the best learning rate using the validation loss.

    Args:
        learning_rates (list[float]): Learning rates to be tested.
        max_epochs (int): Maximum number of epochs to train. Defaults to 20.
        patience (int): Number of epochs with no improvement on validation loss before early stopping.

    Returns:
        tuple[float, int, DataFrame, dict]:
            float: Best learning rate.
            int: Best epoch with the best learning rate.
            DataFrame: Results table from the grid search.
            dict: Histories from the search needed for plotting. 
    """
    rows, histories = [], {}
    for lr in learning_rates:
        print(f"Training with learning rate {lr}")
        set_seed(42)
        model = init_model()
        hist = fit(model, train_loader, val_loader, lr=lr, max_epochs=max_epochs, patience=patience, clip_grad_norm=CLIP)
        best = hist[int(np.nanargmin([h["val_loss"] for h in hist]))]
        histories[lr] = hist
        rows.append(
            {
                "lr": lr,
                "best_epoch": best["epoch"],
                "epochs_run": len(hist),
                "val_loss": best["val_loss"],
                "val_acc": best["val_acc"],
                "val_f1": best["val_f1"],
            }
        )
        
    table = DataFrame(rows)
    best_row = table.loc[table["val_loss"].idxmin()]

    return float(best_row["lr"]), int(best_row["best_epoch"]), table, histories

## Training

In [19]:
def fit(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    lr: float = 1e-3,
    max_epochs: int = 20,
    weight_decay: float = 0.0,
    clip_grad_norm: float | None = None,
    patience: int = 3,
) -> list:
    """
    Trains the model, optionally with early stopping on validation loss.

    If clip_grad_norm is not None, gradients are clipped by global norm after backward.
    We log the pre clipping total gradient norm each epoch.

    Args:
        model (nn.Module): The neural network model to be trained.
        train_loader (DataLoader): DataLoader for the training dataset.
        val_loader (DataLoader): DataLoader for the validation dataset.
        lr (float): Learning rate for the optimizer. Defaults to 1e-3.
        max_epochs (int): Maximum number of epochs to train. Defaults to 20.
        weight_decay (float): Weight decay (L2 regularization) for the optimizer. Defaults to 0.0.
        clip_grad_norm (float | None): Maximum norm for gradient clipping. If None, no clipping is applied. Defaults to None.
        patience (int): Number of epochs with no improvement on validation loss before early stopping. Defaults to 3.

    Returns:
        list: A list of dictionaries containing training and validation metrics for each epoch.
    """
    loss_fn = nn.CrossEntropyLoss()
    optim = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_state = None
    best_val = float("inf")
    bad_epochs = 0

    hist = []
    for epoch in range(1, max_epochs + 1):
        model.train()
        total_loss = 0.0
        n = 0
        correct = 0

        grad_norms = []

        for batch in train_loader:
            x = batch.x.to(device)
            lengths = batch.lengths.to(device)
            y = batch.y.to(device)

            optim.zero_grad(set_to_none=True)
            logits = model(x, lengths)
            loss = loss_fn(logits, y)
            loss.backward()

            total_norm_sq = 0.0
            for p in model.parameters():
                if p.grad is None:
                    continue
                param_norm = p.grad.data.norm(2).item()
                total_norm_sq += param_norm * param_norm
            total_norm = float(total_norm_sq**0.5)
            grad_norms.append(total_norm)

            if clip_grad_norm is not None:
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad_norm)

            optim.step()

            total_loss += loss.item() * y.size(0)
            n += y.size(0)
            correct += (logits.argmax(dim=1) == y).sum().item()

        train_loss = total_loss / max(1, n)
        train_acc = correct / max(1, n)
        val = evaluate(model, val_loader)

        record = {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val["loss"],
            "val_acc": val["acc"],
            "val_f1": val["f1"],
            "grad_norm_mean": float(np.mean(grad_norms))
            if len(grad_norms)
            else float("nan"),
            "grad_norm_p95": float(np.percentile(grad_norms, 95))
            if len(grad_norms)
            else float("nan"),
            "grad_norm_max": float(np.max(grad_norms))
            if len(grad_norms)
            else float("nan"),
        }
        hist.append(record)

        print(
            f"epoch {epoch:02d} | "
            f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
            f"val loss {val['loss']:.4f} acc {val['acc']:.4f} f1 {val['f1']:.4f} | "
            f"grad norm mean {record['grad_norm_mean']:.2f} max {record['grad_norm_max']:.2f} | "
        )

        if patience is not None:
            if val["loss"] < best_val - 1e-6:
                best_val = val["loss"]
                best_state = {
                    k: v.detach().cpu().clone() for k, v in model.state_dict().items()
                }
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    print("Early stopping triggered, restoring best parameters.")
                    if best_state is not None:
                        model.load_state_dict(best_state)
                    break

    if patience is not None and best_state is not None:
        model.load_state_dict(best_state)

    return hist

In [ ]:
def train(model: nn.Module, lr: float, max_epochs: int) -> dict:
    """
    Train a model and evaluate it on validation and test sets.

    Args:
        model (nn.Module): The model to train.
        lr (float): Learning rate used for training.
        max_epochs (int): Maximum number of epochs for training.

    Returns:
        dict: A dictionary containing the model's training history, and evaluation results.
    """
    hist = fit(
        model,
        train_loader,
        val_loader,
        lr=lr,
        max_epochs=max_epochs,
        patience=PATIENCE,
        clip_grad_norm=CLIP,
    )
    val = evaluate(model, val_loader)
    test = evaluate(model, test_loader)
    
    return {
        "hist": hist,
        "val": val,
        "test": test,
        "lr": lr,
        "max_epochs": max_epochs,
    }

## Evaluating

In [21]:
def evaluate(model: nn.Module, loader: DataLoader) -> dict:
    """
    Evaluate the model on a given DataLoader and compute loss, accuracy, and F1 score.

    Args:
        model (nn.Module): The model to evaluate.
        loader (DataLoader): DataLoader for the dataset to evaluate on.

    Returns:
        dict: A dictionary containing the loss, accuracy, F1 score, true labels, and predicted labels.
    """
    model.eval()
    all_y = []
    all_pred = []
    total_loss = 0.0
    n = 0
    loss_fn = nn.CrossEntropyLoss()

    with torch.no_grad():
        for batch in loader:
            x = batch.x.to(device)
            lengths = batch.lengths.to(device)
            y = batch.y.to(device)

            logits = model(x, lengths)
            loss = loss_fn(logits, y)

            pred = logits.argmax(dim=1)
            all_y.append(y.cpu().numpy())
            all_pred.append(pred.cpu().numpy())
            total_loss += loss.item() * y.size(0)
            n += y.size(0)

    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_pred)

    return {
        "loss": total_loss / max(1, n),
        "acc": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, average="macro"),
        "y_true": y_true,
        "y_pred": y_pred,
    }

In [22]:
def count_parameters(model: nn.Module) -> int:
    """
    Count the number of trainable parameters in the model.

    Args:
        model (nn.Module): The model to count parameters for.

    Returns:
        int: The number of trainable parameters.
    """
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [23]:
def plot_lr_search(histories: dict) -> None:
    """
    Plots the values of the grid search of the best learning rate.

    Args:
        histories (dict): Values recorded during the grid search.

    Returns:
        None: Displays the learning rate search plot.
    """
    plt.figure(figsize=(7, 4.5))
    for lr, hist in histories.items():
        plt.plot(
            [h["epoch"] for h in hist],
            [h["val_loss"] for h in hist],
            marker="o",
            label=f"lr = {lr:g}",
        )
    plt.xlabel("Epoch")
    plt.ylabel("Validation loss")
    plt.title("Learning Rate Grid Search", fontsize=13, fontweight="bold")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [24]:
def plot_confusion_matrix(y_true, y_pred, title: str) -> None:
    """
    Plot the confusion matrix for the given true and predicted labels.

    Args:
        y_true (array-like): True labels.
        y_pred (array-like): Predicted labels.
        title (str): Title for the confusion matrix plot.

    Returns:
        None: Displays the confusion matrix plot.
    """
    cm = confusion_matrix(y_true, y_pred)
    _, ax = plt.subplots(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=LABELS.values())
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(title, fontsize=11, fontweight="bold")
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_learning_curves_split(res: dict) -> None:
    """
    Plots train loss vs validation loss for the model.

    Args:
        res (dict): A dictionary containing the model's training history and evaluation results.

    Returns:
        None: Displays the learning curves plot.
    """
    hist = res["hist"]
    epochs = [h["epoch"] for h in hist]
    train_loss = [h["train_loss"] for h in hist]
    val_loss = [h["val_loss"] for h in hist]

    plt.figure(figsize=(6, 4))
    plt.title("Learning Curves", fontsize=13, fontweight="bold")

    plt.plot(epochs, train_loss, marker="o", label="Train Loss", color="steelblue")
    plt.plot(epochs, val_loss, marker="o", label="Validation Loss", color="darkorange")

    plt.xlabel("Epoch")
    plt.ylabel("Cross-entropy loss")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()

    plt.tight_layout()
    plt.show()

## Main - Tuning the hyperparameters, training and evaluating the models

In [ ]:
set_seed(42)

# Tuning the learning rate
best_lr, best_epoch, tuning_table, histories = tune_lr(
    LEARNING_RATES, MAX_EPOCHS, PATIENCE
)
print(tuning_table.to_string(index=False))
plot_lr_search(histories)

Number of trainable parameters:
LSTM: 1986820


In [ ]:
# Initializing the model
set_seed(42)
lstm = init_model()

# Counting the number or parameters
print("Number of trainable parameters:")
print("LSTM:", count_parameters(lstm))

# Training the model
print("Training LSTM...")
res_lstm = train(lstm, best_lr, best_epoch)

In [ ]:
# Evaluating the model
df_results = DataFrame(
    [
        {
            "val_acc": res_lstm["val"]["acc"],
            "val_macro_f1": res_lstm["val"]["f1"],
            "test_acc": res_lstm["test"]["acc"],
            "test_macro_f1": res_lstm["test"]["f1"],
        }
    ]
)

print(df_results.to_string(index=False))

In [ ]:
# Plotting confusion matrices
plot_confusion_matrix(
    res_lstm["val"]["y_true"], res_lstm["val"]["y_pred"], "LSTM - Val Set"
)
plot_confusion_matrix(
    res_lstm["test"]["y_true"], res_lstm["test"]["y_pred"], "LSTM - Test Set"
)

In [ ]:
# Plotting the learning curves
plot_learning_curves_split(res_lstm)